# Parallelism Strategies with vLLM on RunPod

## The Story

You are an ML engineer at a startup building an AI coding assistant. Your journey follows a common trajectory: you start with a single-GPU prototype and hit walls — latency walls, memory walls, throughput walls — and each parallelism strategy is the tool you reach for to break through.

**Act 1 — The Prototype (Single GPU).** You deploy a 7B model on one GPU. It works, but response times are sluggish for real-time chat.

**Act 2 — The Latency Wall (Tensor Parallelism).** Users complain about slow first-token times. You shard the model across GPUs to speed up computation per layer.

**Act 3 — The Memory Wall (Pipeline Parallelism).** Product wants to upgrade to a 70B model. It does not fit on a single node. You split the model's layers across two RunPod machines.

**Act 4 — The Throughput Wall (Data Parallelism).** You launch publicly and get 100 concurrent users. One model replica cannot keep up. You replicate the model across GPUs to handle the load.

**Act 5 — Production (TP x PP x DP).** You combine all three strategies to serve a large model at scale.

Each act introduces a parallelism strategy, shows the exact commands to run it, and measures the impact using standardized latency and throughput metrics.

---

## Metrics Glossary

Before diving in, here are the four metrics we will track throughout this tutorial. Understanding these is essential to evaluating the impact of each parallelism strategy.

**TTFT — Time to First Token (ms)**
The time from when the request is received to when the first token of the response is generated. This corresponds to the prefill (prompt processing) phase. TTFT directly controls perceived responsiveness — it is the "thinking delay" the user sees before output starts streaming.

**TPOT — Time Per Output Token (ms)**
The average time to generate each output token after the first. Calculated as `(total_latency - TTFT) / num_output_tokens`. TPOT reflects the average decode speed. Lower TPOT means faster overall generation.

**ITL — Inter-Token Latency (ms)**
The time between consecutive token emissions during streaming. While TPOT is an average, ITL captures the distribution — including jitter and spikes. High P99 ITL means the stream occasionally stalls, which users perceive as stuttering.

**E2EL — End-to-End Latency (ms)**
The total time from request submission to the last token. `E2EL = TTFT + (num_output_tokens x TPOT)`. This is the bottom-line metric for non-streaming use cases.

### How Each Parallelism Strategy Affects These Metrics

| Metric | Tensor Parallelism (TP) | Pipeline Parallelism (PP) | Data Parallelism (DP) |
|---|---|---|---|
| **TTFT** | Decreases (prefill compute split across GPUs) | Increases slightly (pipeline fill latency) | Unchanged per request, but lower under concurrency (less queueing) |
| **TPOT** | May decrease (faster per-layer compute) or increase (all-reduce overhead) | May increase (inter-stage communication) | Unchanged (each replica runs independently) |
| **ITL** | More consistent (even compute split) | Can be spiky (pipeline bubble stalls) | More consistent under load (less contention) |
| **Throughput** | Marginal improvement | Marginal improvement | Near-linear scaling with replica count |

---

## Prerequisites

- A RunPod account with GPU pod access
- A HuggingFace token (for gated models like Llama)
- Familiarity with SSH and basic Linux commands

---

## 1. RunPod Setup

### 1.1 Single-Node Pod (for TP, DP, single-node PP)

Provision a multi-GPU pod:

1. Navigate to **Pods > Deploy**.
2. Select a **4xA100 (80 GB)** or **4xA6000 (48 GB)** instance.
3. Pick the template `runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04` (or a recent PyTorch + CUDA devel image).
4. Under **Customize Deployment**, expose port **8000** via HTTP (for the OpenAI-compatible server).
5. Deploy and connect via SSH or the web terminal.

### 1.2 Multi-Node Setup (for cross-node PP)

RunPod offers two options for multi-node:

**Option A — Instant Clusters (recommended)**

1. Go to **Clusters** in the RunPod console.
2. Create a 2-node cluster with **1xA100 (or more) per node**.
3. RunPod provisions both nodes on the same private network with high-speed interconnects (InfiniBand / RoCE v2, up to 3200 Gbps).
4. You get SSH access to both nodes. Environment variables like `RUNPOD_POD_ID`, node IPs, and hostnames are pre-configured.

**Option B — Two Separate GPU Pods (manual networking)**

1. Deploy two single-GPU (or multi-GPU) pods in the **same region**.
2. Note down the internal/public IPs of both pods.
3. Ensure both pods can reach each other on port **6379** (Ray) and ports **8000-8100** (vLLM / NCCL). You may need to configure TCP proxy or use RunPod's public IPs with exposed ports.

> Instant Clusters are strongly preferred because they come with private network connectivity out of the box. Two separate pods communicate over the public internet, which adds latency to NCCL operations and makes cross-node TP impractical. PP is more tolerant of network latency since it only passes activations between pipeline stages.

---

## 2. Environment Setup with `uv`

Run these steps on **every node** that will participate in serving.

### 2.1 Install `uv`

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
source $HOME/.local/bin/env
```

Verify:

```bash
uv --version
```

### 2.2 Create a Virtual Environment and Install Dependencies

```bash
mkdir -p ~/vllm_parallelism && cd ~/vllm_parallelism

uv venv .venv --python 3.11
source .venv/bin/activate

uv pip install vllm
uv pip install "ray[default]"
```

### 2.3 Verify the Setup

```bash
python -c "import torch; print(f'GPUs visible: {torch.cuda.device_count()}')"
python -c "import vllm; print(f'vLLM version: {vllm.__version__}')"
ray --version
```

### 2.4 Set HuggingFace Token

```bash
export HF_TOKEN="hf_your_token_here"
```

---

## 3. Model Selection

| Model | Params | FP16 VRAM | Use Case |
|---|---|---|---|
| `Qwen/Qwen2.5-7B-Instruct` | 7B | ~14 GB | Fits on 1 GPU. Good for showing TP/DP speedups without memory pressure. |
| `Qwen/Qwen2.5-14B-Instruct` | 14B | ~28 GB | Needs TP=2 on A6000 (48 GB). Clean demo of "model won't fit without parallelism". |
| `meta-llama/Llama-3.1-70B-Instruct` | 70B | ~140 GB | Requires TP>=2 on A100-80GB. Best for multi-node PP demos. |

```bash
export MODEL="Qwen/Qwen2.5-7B-Instruct"
```

---

## 4. Act 1 — The Prototype (Baseline, Single GPU)

You deploy your first model on a single GPU. Everything works, but you need to establish baseline metrics before optimizing.

### 4.1 Serve the Model on 1 GPU

```bash
vllm serve $MODEL \
  --tensor-parallel-size 1 \
  --port 8000
```

Test it in a second terminal:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "Write a Python function to check if a string is a palindrome."}],
    "max_tokens": 200
  }' | python -m json.tool
```

### 4.2 Measure Baseline Metrics

Run the `vllm bench serve` benchmark against the running server. This reports TTFT, TPOT, ITL, and E2EL:

```bash
vllm bench serve \
  --model $MODEL \
  --base-url http://localhost:8000 \
  --backend openai-chat \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 128 \
  --num-prompts 100 \
  --request-rate 5 \
  --percentile-metrics ttft,tpot,itl,e2el \
  --metric-percentiles 50,90,99 \
  --save-result \
  --output-json results_baseline.json
```

Flag breakdown:

- `--percentile-metrics ttft,tpot,itl,e2el` — Report percentiles for all four key metrics.
- `--metric-percentiles 50,90,99` — Report the median (P50), P90, and P99 for each metric.
- `--save-result --output-json results_baseline.json` — Save the full results to a JSON file for later comparison.
- `--request-rate 5` — Send 5 requests per second (simulates moderate traffic).
- `--dataset-name random` — Use synthetic random prompts so the benchmark is reproducible.

Expected output format:

```
============ Serving Benchmark Result ============
Successful requests:                     100
Benchmark duration (s):                  42.15
Total input tokens:                      51200
Total generated tokens:                  12800
Request throughput (req/s):              2.37
Output token throughput (tok/s):         303.68
Total token throughput (tok/s):          1518.39
---------------Time to First Token----------------
Mean TTFT (ms):                          85.23
Median TTFT (ms):                        78.40
P90 TTFT (ms):                           112.50
P99 TTFT (ms):                           145.20
-----Time per Output Token (excl. 1st token)------
Mean TPOT (ms):                          12.45
Median TPOT (ms):                        12.10
P90 TPOT (ms):                           14.30
P99 TPOT (ms):                           18.90
---------------Inter-token Latency----------------
Mean ITL (ms):                           12.80
Median ITL (ms):                         11.90
P90 ITL (ms):                            15.20
P99 ITL (ms):                            25.40
--------------End-to-end Latency------------------
Mean E2EL (ms):                          1680.50
Median E2EL (ms):                        1620.30
P90 E2EL (ms):                           1950.00
P99 E2EL (ms):                           2340.00
==================================================
```

> The numbers above are illustrative. Your actual numbers will depend on GPU type, model, and workload. The key is to record these as the baseline you will compare against.

### 4.3 Record GPU Memory Baseline

```bash
nvidia-smi --query-gpu=index,memory.used,memory.total,utilization.gpu --format=csv
```

Note the VRAM used by the model on GPU 0. All other GPUs should be idle.

---

## 5. Act 2 — The Latency Wall (Tensor Parallelism)

### The Problem

Users of your coding assistant report that the model "takes too long to start responding." Looking at your baseline metrics, the TTFT is 85ms at P50, but under heavier load it spikes to 145ms at P99. The prefill phase (processing the user's prompt) is compute-bound — the GPU is doing all the matrix multiplications for all layers sequentially.

### The Solution: Tensor Parallelism

TP shards each layer's weight matrices across GPUs. Every GPU holds a **slice of every layer**. During the forward pass, each GPU computes its portion and the results are combined via all-reduce. This reduces per-GPU memory and can lower latency for compute-bound workloads.

### 5.1 TP=2 (Model Split Across 2 GPUs)

Stop the baseline server (`Ctrl+C`), then:

```bash
vllm serve $MODEL \
  --tensor-parallel-size 2 \
  --port 8000
```

Test it:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "Explain the difference between a mutex and a semaphore with code examples."}],
    "max_tokens": 250
  }' | python -m json.tool
```

Benchmark:

```bash
vllm bench serve \
  --model $MODEL \
  --base-url http://localhost:8000 \
  --backend openai-chat \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 128 \
  --num-prompts 100 \
  --request-rate 5 \
  --percentile-metrics ttft,tpot,itl,e2el \
  --metric-percentiles 50,90,99 \
  --save-result \
  --output-json results_tp2.json
```

### 5.2 TP=4 (Model Split Across 4 GPUs)

```bash
vllm serve $MODEL \
  --tensor-parallel-size 4 \
  --port 8000
```

Test it:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "How does NCCL all-reduce work across GPUs connected via NVLink?"}],
    "max_tokens": 200
  }' | python -m json.tool
```

Benchmark:

```bash
vllm bench serve \
  --model $MODEL \
  --base-url http://localhost:8000 \
  --backend openai-chat \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 128 \
  --num-prompts 100 \
  --request-rate 5 \
  --percentile-metrics ttft,tpot,itl,e2el \
  --metric-percentiles 50,90,99 \
  --save-result \
  --output-json results_tp4.json
```

### 5.3 What to Observe

Compare the JSON result files across configurations:

```bash
python3 -c "
import json

for fname, label in [
    ('results_baseline.json', 'TP=1'),
    ('results_tp2.json', 'TP=2'),
    ('results_tp4.json', 'TP=4'),
]:
    with open(fname) as f:
        d = json.load(f)
    print(f'{label}:')
    print(f'  TTFT  mean={d[\"mean_ttft_ms\"]:.1f}ms  p99={d[\"p99_ttft_ms\"]:.1f}ms')
    print(f'  TPOT  mean={d[\"mean_tpot_ms\"]:.1f}ms  p99={d[\"p99_tpot_ms\"]:.1f}ms')
    print(f'  ITL   mean={d[\"mean_itl_ms\"]:.1f}ms  p99={d[\"p99_itl_ms\"]:.1f}ms')
    print(f'  Throughput: {d[\"output_throughput\"]:.1f} tok/s')
    print()
"
```

Expected patterns:

- **TTFT drops** as TP increases — the prefill computation is split across more GPUs, so each does less work.
- **TPOT may slightly increase** at TP=4 due to all-reduce communication overhead dominating the decode step for a relatively small model.
- **ITL becomes more consistent** (lower P99/P50 ratio) because compute is balanced across GPUs.
- **VRAM per GPU decreases** proportionally — check with `nvidia-smi`.

Monitor GPU memory to confirm the split:

```bash
watch -n 1 nvidia-smi --query-gpu=index,memory.used,memory.total,utilization.gpu --format=csv
```

With TP=4, all 4 GPUs should show similar VRAM usage (roughly 1/4 of the full model each) and similar utilization.

---

## 6. Act 3 — The Memory Wall (Pipeline Parallelism)

### The Problem

Product wants to upgrade from the 7B model to a 70B model for better code quality. The 70B model needs ~140 GB of VRAM in FP16. Even a single node with 4xA100-80GB (320 GB total) could fit it with TP=4, but what if you only have 2xA100-80GB? Or what if the model is even larger? You need to spread layers across multiple machines.

### The Solution: Pipeline Parallelism

PP assigns different **groups of consecutive layers** to different GPUs (or nodes). GPU 0 runs layers 0-N/k, GPU 1 runs layers N/k+1-2N/k, and so on. Data flows sequentially through the pipeline. Unlike TP, there is no all-reduce — only point-to-point activation transfers between pipeline stages.

### 6.1 Single-Node PP

#### PP=2 (Layers Split Across 2 GPUs)

```bash
vllm serve $MODEL \
  --pipeline-parallel-size 2 \
  --port 8000
```

Test it:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "What is the pipeline bubble problem and why does it hurt single-request latency?"}],
    "max_tokens": 200
  }' | python -m json.tool
```

Benchmark:

```bash
vllm bench serve \
  --model $MODEL \
  --base-url http://localhost:8000 \
  --backend openai-chat \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 128 \
  --num-prompts 100 \
  --request-rate 5 \
  --percentile-metrics ttft,tpot,itl,e2el \
  --metric-percentiles 50,90,99 \
  --save-result \
  --output-json results_pp2.json
```

#### PP=4

```bash
vllm serve $MODEL \
  --pipeline-parallel-size 4 \
  --port 8000
```

Test it:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "How does micro-batching reduce pipeline bubble overhead in pipeline-parallel inference?"}],
    "max_tokens": 200
  }' | python -m json.tool
```

Benchmark:

```bash
vllm bench serve \
  --model $MODEL \
  --base-url http://localhost:8000 \
  --backend openai-chat \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 128 \
  --num-prompts 100 \
  --request-rate 5 \
  --percentile-metrics ttft,tpot,itl,e2el \
  --metric-percentiles 50,90,99 \
  --save-result \
  --output-json results_pp4.json
```

#### Combining TP + PP on a Single Node

On a 4-GPU node, you can combine both:

```bash
# TP=2 x PP=2 = 4 GPUs total
vllm serve Qwen/Qwen2.5-14B-Instruct \
  --tensor-parallel-size 2 \
  --pipeline-parallel-size 2 \
  --port 8000
```

Test it:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-14B-Instruct",
    "messages": [{"role": "user", "content": "Why do production LLM deployments combine tensor parallelism with pipeline parallelism instead of using just one?"}],
    "max_tokens": 250
  }' | python -m json.tool
```

Benchmark:

```bash
vllm bench serve \
  --model Qwen/Qwen2.5-14B-Instruct \
  --base-url http://localhost:8000 \
  --backend openai-chat \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 128 \
  --num-prompts 100 \
  --request-rate 5 \
  --percentile-metrics ttft,tpot,itl,e2el \
  --metric-percentiles 50,90,99 \
  --save-result \
  --output-json results_tp2_pp2.json
```

#### What to Observe

- **TTFT increases slightly** compared to TP — the first pipeline stage must pass activations to subsequent stages before the first token can be emitted. This is the pipeline fill latency.
- **ITL can be spiky** — the pipeline bubble problem means later stages idle periodically, causing uneven token emission.
- **VRAM distribution is uneven** — the first and last stages carry extra memory for the embedding layer and LM head. Check with `nvidia-smi`.
- **Throughput under high load can improve** — with many requests in flight, the pipeline stages overlap computation, increasing overall throughput.

### 6.2 Multi-Node PP with Two RunPod Instances

This is the key section: running PP across two separate RunPod machines using Ray to form a distributed cluster.

#### Architecture Overview

```
+---------------------+         +---------------------+
|   Node 0 (Head)     |         |   Node 1 (Worker)   |
|                     |  NCCL   |                     |
|  Layers 0 .. N/2    |<------->|  Layers N/2+1 .. N  |
|                     |         |                     |
|  Ray Head           |  6379   |  Ray Worker         |
|  vLLM Server :8000  |         |                     |
+---------------------+         +---------------------+
```

#### Step 1: Provision Two RunPod Instances

Using Instant Clusters:

1. Go to **Clusters** in RunPod console.
2. Select **2 nodes**, each with **1xA100-80GB** (or more GPUs per node for TPxPP combos).
3. Deploy. RunPod assigns private IPs and sets up the interconnect.
4. SSH into both nodes. Note the private IPs:
   - Node 0 (head): e.g., `10.0.0.1`
   - Node 1 (worker): e.g., `10.0.0.2`

#### Step 2: Install Environment on Both Nodes

Run on **both** Node 0 and Node 1:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
source $HOME/.local/bin/env

mkdir -p ~/vllm_parallelism && cd ~/vllm_parallelism
uv venv .venv --python 3.11
source .venv/bin/activate

uv pip install vllm "ray[default]"

export HF_TOKEN="hf_your_token_here"
```

#### Step 3: Start the Ray Cluster

**On Node 0 (head node):**

```bash
export HEAD_IP=$(hostname -I | awk '{print $1}')
echo "Head IP: $HEAD_IP"

ray start --head \
  --port=6379 \
  --dashboard-host=0.0.0.0 \
  --dashboard-port=8265
```

**On Node 1 (worker node):**

```bash
export HEAD_IP="10.0.0.1"  # Replace with actual head node IP

ray start --address="${HEAD_IP}:6379"
```

#### Step 4: Verify the Ray Cluster

```bash
ray status
python -c "import ray; ray.init(); print([n for n in ray.nodes() if n['Alive']])"
```

#### Step 5: Download the Model on Both Nodes

```bash
# Run on BOTH nodes (or use shared Network Storage)
python -c "
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-7B-Instruct')
"
```

#### Step 6: Launch vLLM with Multi-Node PP

Run on the **head node only**:

```bash
export VLLM_HOST_IP=$(hostname -I | awk '{print $1}')

vllm serve Qwen/Qwen2.5-7B-Instruct \
  --tensor-parallel-size 1 \
  --pipeline-parallel-size 2 \
  --distributed-executor-backend ray \
  --port 8000
```

Test it:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "Explain pipeline parallelism in one sentence."}],
    "max_tokens": 100
  }' | python -m json.tool
```

For multi-GPU nodes (e.g., 4 GPUs per node), combine TP and PP:

```bash
# 4 GPUs per node x 2 nodes = 8 GPUs total
vllm serve meta-llama/Llama-3.1-70B-Instruct \
  --tensor-parallel-size 4 \
  --pipeline-parallel-size 2 \
  --distributed-executor-backend ray \
  --port 8000
```

Test it:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "meta-llama/Llama-3.1-70B-Instruct",
    "messages": [{"role": "user", "content": "Describe the trade-offs between inter-node and intra-node communication in distributed LLM inference."}],
    "max_tokens": 250
  }' | python -m json.tool
```

#### Step 7: Benchmark Multi-Node PP

```bash
vllm bench serve \
  --model Qwen/Qwen2.5-7B-Instruct \
  --base-url http://localhost:8000 \
  --backend openai-chat \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 128 \
  --num-prompts 100 \
  --request-rate 5 \
  --percentile-metrics ttft,tpot,itl,e2el \
  --metric-percentiles 50,90,99 \
  --save-result \
  --output-json results_multinode_pp2.json
```

Compare the TTFT from this run against the single-node PP=2 result. Multi-node PP will show higher TTFT and ITL due to network latency between nodes, but the model fits — which is the point.

#### Step 8: Verify Cross-Node GPU Usage

On Node 0:

```bash
nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv
```

On Node 1:

```bash
nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv
```

Each node should show roughly half the model's memory footprint.

#### Troubleshooting Multi-Node

| Problem | Fix |
|---|---|
| `ray status` shows only 1 node | Check that the worker node can reach `HEAD_IP:6379`. Verify firewall rules and security groups. |
| NCCL timeout errors | Set `export NCCL_SOCKET_IFNAME=eth0` (or your network interface) on both nodes. For Instant Clusters with InfiniBand, set `NCCL_IB_HCA=mlx5`. |
| `VLLM_HOST_IP` errors | Explicitly export `VLLM_HOST_IP` to the node's private IP on the head node before starting vLLM. |
| Model not found on worker | Ensure the model is downloaded on all nodes at the same path, or use shared storage. |
| Slow cross-node performance | Check NCCL transport with `NCCL_DEBUG=TRACE`. If you see `NET/Socket`, you are using TCP (slow). `NET/IB/GDRDMA` means InfiniBand is active (fast). |

#### Cleaning Up

```bash
# On both nodes
ray stop
```

---

## 7. Act 4 — The Throughput Wall (Data Parallelism)

### The Problem

You launch your coding assistant publicly. Within a week you have 100 concurrent users. The 7B model on TP=2 handles individual requests fast (good TTFT and TPOT), but under high concurrency, requests queue up. The server's throughput caps at ~300 tok/s, and P99 TTFT spikes to 500ms+ because new requests wait for in-flight requests to free up KV cache slots.

### The Solution: Data Parallelism

DP replicates the entire model on each GPU (or group of GPUs) and splits incoming requests across replicas. Each replica independently processes its share of requests. No gradient synchronization is needed (this is inference, not training), so DP scales throughput almost linearly with the number of replicas.

### 7.1 DP=2 (Two Model Replicas)

```bash
vllm serve $MODEL \
  --data-parallel-size 2 \
  --tensor-parallel-size 1 \
  --port 8000
```

Test it:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "How does data parallelism differ during inference compared to training?"}],
    "max_tokens": 200
  }' | python -m json.tool
```

### 7.2 DP=2 x TP=2 (Four GPUs Total)

```bash
vllm serve $MODEL \
  --data-parallel-size 2 \
  --tensor-parallel-size 2 \
  --port 8000
```

Test it:

```bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "Write a Python function to calculate the Fibonacci sequence using memoization."}],
    "max_tokens": 250
  }' | python -m json.tool
```

### 7.3 Benchmarking DP Under Load

DP shines under high concurrency. The key is to increase `--request-rate` and `--max-concurrency` to simulate realistic multi-user traffic:

```bash
# Benchmark with DP=1 (baseline under heavy load)
vllm serve $MODEL --tensor-parallel-size 1 --port 8000 &
sleep 30

vllm bench serve \
  --model $MODEL \
  --base-url http://localhost:8000 \
  --backend openai-chat \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 128 \
  --num-prompts 200 \
  --request-rate 20 \
  --max-concurrency 50 \
  --percentile-metrics ttft,tpot,itl,e2el \
  --metric-percentiles 50,90,99 \
  --save-result \
  --output-json results_dp1_highload.json

kill %1
sleep 5
```

```bash
# Benchmark with DP=2 under the same heavy load
vllm serve $MODEL --data-parallel-size 2 --tensor-parallel-size 1 --port 8000 &
sleep 30

vllm bench serve \
  --model $MODEL \
  --base-url http://localhost:8000 \
  --backend openai-chat \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 128 \
  --num-prompts 200 \
  --request-rate 20 \
  --max-concurrency 50 \
  --percentile-metrics ttft,tpot,itl,e2el \
  --metric-percentiles 50,90,99 \
  --save-result \
  --output-json results_dp2_highload.json

kill %1
```

### 7.4 What to Observe

Compare the two JSON files:

```bash
python3 -c "
import json

for fname, label in [
    ('results_dp1_highload.json', 'DP=1 (high load)'),
    ('results_dp2_highload.json', 'DP=2 (high load)'),
]:
    with open(fname) as f:
        d = json.load(f)
    print(f'{label}:')
    print(f'  Throughput: {d[\"output_throughput\"]:.1f} tok/s')
    print(f'  TTFT  p50={d[\"median_ttft_ms\"]:.1f}ms  p99={d[\"p99_ttft_ms\"]:.1f}ms')
    print(f'  TPOT  p50={d[\"median_tpot_ms\"]:.1f}ms  p99={d[\"p99_tpot_ms\"]:.1f}ms')
    print(f'  ITL   p50={d[\"median_itl_ms\"]:.1f}ms  p99={d[\"p99_itl_ms\"]:.1f}ms')
    print(f'  E2EL  p50={d[\"median_e2el_ms\"]:.1f}ms  p99={d[\"p99_e2el_ms\"]:.1f}ms')
    print()
"
```

Expected patterns:

- **Throughput roughly doubles** from DP=1 to DP=2 (two independent replicas processing requests in parallel).
- **P99 TTFT drops significantly** — with DP=2, requests are distributed across replicas, so queue wait times shrink.
- **TPOT stays similar** per request (each replica runs the same model independently).
- **P99 ITL drops** — less contention means fewer scheduling stalls.
- **VRAM**: Each GPU loads the full model. Total VRAM usage is `model_size x num_replicas`.

---

## 8. Act 5 — Production (Putting It All Together)

In production, you combine strategies based on your constraints:

| Constraint | Strategy |
|---|---|
| Model fits on 1 GPU, need lower latency | TP (start with TP=2) |
| Model does not fit on 1 GPU | PP or TP (whichever the GPU count supports) |
| Model does not fit on 1 node | Multi-node PP (with intra-node TP) |
| Need higher throughput under load | DP (replicate model, split traffic) |
| Production large-model serving | TP x PP x DP combined |

The constraint: **TP x PP x DP = Total GPU count**

For example, on a 2-node cluster with 4 GPUs per node (8 GPUs total):

- TP=4 x PP=2 x DP=1 = 8 GPUs — One large-model replica across 2 nodes
- TP=2 x PP=1 x DP=4 = 8 GPUs — Four small-model replicas, each using TP=2
- TP=2 x PP=2 x DP=2 = 8 GPUs — Two replicas, each spanning 2 nodes with TP=2

---

## 9. Automated Benchmark Sweep

Create a script to run all single-node configurations and collect comparable metrics:

```bash
#!/bin/bash
# benchmark_sweep.sh
# Run from the single-node 4-GPU pod

MODEL="Qwen/Qwen2.5-7B-Instruct"
BENCH_ARGS="--backend openai-chat --dataset-name random --random-input-len 512 --random-output-len 128 --num-prompts 100 --request-rate 5 --percentile-metrics ttft,tpot,itl,e2el --metric-percentiles 50,90,99 --save-result"

run_config() {
    local label=$1
    local output_file=$2
    shift 2
    local serve_args="$@"

    echo "============================================"
    echo ">>> Starting: $label"
    echo ">>> Serve args: $serve_args"
    echo "============================================"

    vllm serve $MODEL $serve_args --port 8000 &
    local pid=$!

    echo "Waiting for server..."
    for i in $(seq 1 60); do
        if curl -s http://localhost:8000/v1/models > /dev/null 2>&1; then
            echo "Server ready after ${i}s"
            break
        fi
        sleep 1
    done

    vllm bench serve \
        --model $MODEL \
        --base-url http://localhost:8000 \
        $BENCH_ARGS \
        --output-json "$output_file"

    kill $pid 2>/dev/null
    wait $pid 2>/dev/null
    sleep 5
    echo ""
}

run_config "Baseline (TP=1)" "sweep_tp1.json" "--tensor-parallel-size 1"
run_config "TP=2" "sweep_tp2.json" "--tensor-parallel-size 2"
run_config "TP=4" "sweep_tp4.json" "--tensor-parallel-size 4"
run_config "PP=2" "sweep_pp2.json" "--pipeline-parallel-size 2"
run_config "PP=4" "sweep_pp4.json" "--pipeline-parallel-size 4"
run_config "TP=2 PP=2" "sweep_tp2_pp2.json" "--tensor-parallel-size 2 --pipeline-parallel-size 2"
run_config "DP=2" "sweep_dp2.json" "--data-parallel-size 2 --tensor-parallel-size 1"
run_config "DP=2 TP=2" "sweep_dp2_tp2.json" "--data-parallel-size 2 --tensor-parallel-size 2"

echo ""
echo "============================================"
echo "All benchmarks complete. Summary:"
echo "============================================"

python3 << 'PYEOF'
import json, glob, os

files = sorted(glob.glob('sweep_*.json'))
header = f"{'Config':<16} {'Tput(tok/s)':>12} {'TTFT p50':>10} {'TTFT p99':>10} {'TPOT p50':>10} {'TPOT p99':>10} {'ITL p99':>10} {'E2EL p99':>10}"
print(header)
print("-" * len(header))

for f in files:
    label = os.path.basename(f).replace('sweep_', '').replace('.json', '').upper()
    with open(f) as fh:
        d = json.load(fh)
    print(f"{label:<16} {d['output_throughput']:>10.1f}/s {d['median_ttft_ms']:>8.1f}ms {d['p99_ttft_ms']:>8.1f}ms {d['median_tpot_ms']:>8.1f}ms {d['p99_tpot_ms']:>8.1f}ms {d['p99_itl_ms']:>8.1f}ms {d['p99_e2el_ms']:>8.1f}ms")
PYEOF
```

```bash
chmod +x benchmark_sweep.sh
./benchmark_sweep.sh
```

### Expected Results Pattern

| Config | Throughput | TTFT p50 | TTFT p99 | TPOT p50 | ITL p99 | Best For |
|---|---|---|---|---|---|---|
| TP=1 (baseline) | Baseline | Baseline | Baseline | Baseline | Baseline | Single-user, small models |
| TP=2 | Similar | Lower | Lower | Similar | Similar | Reducing latency |
| TP=4 | May drop | Lowest | Lowest | May increase | Similar | Large models, latency-critical |
| PP=2 | Similar | Higher | Higher | Higher | Higher | Fitting larger models |
| PP=4 | Similar | Highest | Highest | Higher | Highest | Very large models |
| TP=2 x PP=2 | Similar | Moderate | Moderate | Moderate | Moderate | Production large-model serving |
| DP=2 | ~2x | Same | Much lower* | Same | Much lower* | High-throughput serving |
| DP=2 x TP=2 | ~2x | Lower | Much lower* | Similar | Much lower* | Throughput + latency |

*Under high concurrency. At low concurrency, DP shows no improvement since there is no queueing.

---

## 10. Prometheus Metrics (Live Monitoring)

Beyond benchmarking, vLLM exposes real-time Prometheus metrics on the `/metrics` endpoint. You can scrape these while the server is running:

```bash
curl -s http://localhost:8000/metrics | grep -E "^vllm:(time_to_first_token|inter_token_latency|e2e_request_latency|num_requests)"
```

Key metrics available:

- `vllm:time_to_first_token_seconds` — Histogram of TTFT values
- `vllm:inter_token_latency_seconds` — Histogram of ITL values
- `vllm:e2e_request_latency_seconds` — Histogram of end-to-end latency
- `vllm:request_prefill_time_seconds` — Time spent in prefill phase
- `vllm:request_decode_time_seconds` — Time spent in decode phase
- `vllm:num_requests_running` — Current in-flight requests
- `vllm:num_requests_waiting` — Requests queued (high values here signal the need for DP)

---

## 11. GPU Monitoring Cheat Sheet

```bash
# Continuous monitoring
watch -n 1 nvidia-smi

# Compact CSV output
nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu,temperature.gpu \
  --format=csv -l 1

# Process-level GPU usage
nvidia-smi pmon -s um -c 1
```

What to look for per configuration:

- **TP**: All GPUs show similar VRAM usage and similar GPU utilization.
- **PP**: GPUs show different VRAM amounts (first/last stage slightly higher due to embedding/LM head). Utilization may be uneven due to pipeline bubbles.
- **DP**: Each GPU shows full model VRAM. All GPUs at high utilization under load.

---

## 12. Quick Reference

### vLLM Serve Flags

| Flag | Description |
|---|---|
| `--tensor-parallel-size N` | Shard every layer across N GPUs |
| `--pipeline-parallel-size N` | Split layers into N sequential stages |
| `--data-parallel-size N` | Run N independent model replicas |
| `--distributed-executor-backend ray` | Use Ray for multi-node coordination |
| `--port 8000` | Port for the OpenAI-compatible API |
| `--gpu-memory-utilization 0.9` | Fraction of GPU memory vLLM can use for KV cache |

### vLLM Bench Serve Flags (Metrics Collection)

| Flag | Description |
|---|---|
| `--percentile-metrics ttft,tpot,itl,e2el` | Which metrics to report percentiles for |
| `--metric-percentiles 50,90,99` | Which percentiles to calculate (default: 99 only) |
| `--save-result` | Save results to file |
| `--output-json FILE` | Path for the JSON results file |
| `--request-rate N` | Requests per second to send |
| `--max-concurrency N` | Maximum concurrent requests |
| `--dataset-name random` | Use synthetic random prompts |
| `--random-input-len N` | Input token length for random dataset |
| `--random-output-len N` | Output token length for random dataset |
| `--goodput ttft:MS tpot:MS` | Define SLO thresholds for goodput calculation |
| `--ignore-eos` | Ignore EOS token to force exact output length |

---

## 13. Cleanup

```bash
pkill -f "vllm serve"
ray stop
deactivate
```

If using RunPod Instant Clusters, terminate the cluster from the RunPod console when done to stop billing.

---

## Summary: The Story Arc

1. **Prototype** — Single GPU, baseline metrics recorded. TTFT is okay, throughput is limited.
2. **Latency Wall** — TP splits layers across GPUs. TTFT drops. Users see faster first responses.
3. **Memory Wall** — PP splits layer groups across nodes. Models that cannot fit on one machine now run.
4. **Throughput Wall** — DP replicates the model. Throughput scales linearly. P99 latencies drop under load.
5. **Production** — Combine TP x PP x DP to serve large models at scale with acceptable latency and throughput.

Each strategy addresses a different bottleneck. Measuring TTFT, TPOT, ITL, and E2EL at every step tells you exactly which wall you have hit and whether the strategy you applied actually moved the needle.